<h1 style="color:orange">EXERCISE CLASS 4 </h1>

## EXERCISE 2

The data in the `LPBF.csv` file represents the deviation from the nominal shape (in mm²) of the area of 300 consecutive layers printed using laser powder bed fusion (LPBF) technology. 

1. Check if data are autocorrelated. If they are, propose a strategy to get rid of the data auto-correlation.
2. Design an I-MR control chart. Assume an assignable cause is found for any out-of-control points.
3. Re-design the I-MR control chart with probability limits (i.e., use the true distribution of both statistics) with $\alpha=0.012$. With regard to the MR chart, use the Box-Cox data transformation. Assume an assignable cause is found for any out-of-control points.



### 1. Check if data are autocorrelated. If they are, propose a strategy to get rid of the data auto-correlation.

In [ ]:
# Import the necessary libraries
import qdatoolkit as qda
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from scipy import stats
import qdatoolkit as qda


# Import the dataset
data = pd.read_csv('../Dataset/LPBF.csv')

data.head()

In [ ]:
#Time series plot
plt.figure(figsize=(15, 5))
plt.plot(data, 'o-')
plt.title('Time series plot')
plt.xlabel('Index')
plt.ylabel('Value')
plt.show()

In [ ]:
# Perform the runs test, plot the acf and plot the pacf with qda-toolkit
_ = qda.Assumptions(data).independence()  

In [ ]:
# Perform Bartlett's test at lag 1
_ = qda.Assumptions(data).independence(plotit = False, ac_test='bartlett', lag=1)

In [ ]:
# Perform LBQ test for autocorrelation at lag 4
_ = qda.Assumptions(data).independence(plotit = False, ac_test='lbq', lag=4)

> The data are auto-correlated, check if:
> 1. Data gapping is suitable to get rid of the data autocorrelation. 
> 2. Data batching is suitable to get rid of the data autocorrelation. 

#### 1.1 Gapping

> Let's try data gapping first. 

In [ ]:
#Gapping 
gap_size= 6 # this is just an example, you can try different gapping intervals
gap_num= int(len(data)/gap_size)

gap_data= np.zeros((gap_num))
for i in range (gap_num):
    gap_data[i]=data['mm^2'][i*6]

> Alternatively, use the built-in array manipulation functionalities:

In [ ]:
# Take one data point every 6
gap_data = data['mm^2'][::gap_size]

> Let's analyze data after gapping

In [ ]:
#Time series plot and runs test
plt.plot(gap_data, 'o-')
plt.title('Time series plot')
plt.xlabel('Index')
plt.ylabel('Value')
plt.show()

In [ ]:
# Runs test, ACF and PACF for gapped data
_ = qda.Assumptions(gap_data).independence()

In [ ]:
# Perform Bartlett's test at lag 1 for gapped data
_ = qda.Assumptions(gap_data).independence(plotit = False, ac_test='bartlett', lag=1)

#### 1.2 Batching

> Let's try with batching. Create 50 subgroups of size 6. 

In [ ]:
# Batching
batch_size = 6
batch_num = int(len(data)/batch_size)

j=0
batch_data = np.zeros((batch_num))
for i in range (batch_num):
    batch_data[i]=np.sum(data['mm^2'][j:j+batch_size])/batch_size
    j=j+batch_size

In [ ]:
# Alternative method
# Create a new column in the dataframe with the corresponding batch number
data['Batch'] = np.repeat(np.arange(1, batch_num+1), batch_size)

In [ ]:
# Store the batch means in a new dataframe
batch_data = data.groupby('Batch').mean()

> Let's analyze data after batching

In [ ]:
#Time series plot
plt.plot(batch_data, 'o-')
plt.title('Time series plot')
plt.xlabel('Index')
plt.ylabel('Value')
plt.show()

> Check if the data are random.

In [ ]:
# Runs test, ACF and PACF for gapped data
_ = qda.Assumptions(batch_data).independence()

> Now check for autocorrelation.

In [ ]:
# Perform Bartlett's test at lag 1 for gapped data
_ = qda.Assumptions(batch_data).independence(plotit = False, ac_test='bartlett', lag=1)

> Both gapping and batching can eliminate data autocorrelation. However, based on the results of the independence tests, we have greater confidence in the independence of the data when gapping is applied. Therefore, we will proceed with the gapped data.

In [ ]:
# Store the gapped data
data = pd.DataFrame(gap_data, columns=['mm^2'])
data = data.reset_index(drop=True)


### 2. Design an I-MR control chart

In [ ]:
# Check for normality
_ = qda.Assumptions(data['mm^2']).normality()

In [ ]:
# Design the I-MR chart
data_IMR = qda.ControlCharts.IMR(data, 'mm^2')

> The process is in control.

### 3. Re-design the I-MR control chart with probability limits (i.e., use the true distribution of both statistics) with $\alpha=0.012$. With regard to the MR chart, use the Box-Cox data transformation.

In [ ]:
# Find the new K value 
alpha = 0.012
K_alpha = stats.norm.ppf(1-alpha/2)
print('New K value = %.3f' % K_alpha)

In [ ]:
# Design the I control chart with the new K value
I = qda.ControlCharts.I(data, 'mm^2', K = K_alpha)

> The I chart signals one out-of-control observation. Before recomputing control limits, let's check the MR chart.

> We can apply a known transformation (lambda = 0.4) to the MR values and then compute probabilistic control limits.


In [ ]:
# Extract the 'MR' column   
MR = pd.DataFrame(I['MR'])
# Transform the data
MR['MR_transformed'] = MR['MR'].transform(lambda x: ((x**0.4)))

In [ ]:
# Let's check normality
_ = qda.Assumptions(MR['MR_transformed']).normality()

In [ ]:
I_MR = qda.ControlCharts.I(MR, 'MR_transformed', K = K_alpha)

> The I chart on MR signals one OOC observation. 

> We must remove the out-of-control observations and recompute control limits. First, let's find the indexes of OOC points in the two charts.

In [ ]:
# Let's find the array of indexes corresponding to OOC points in the I chart for mm^2

# Find the index of the I_TEST1 column different from NaN
OOC_idx = np.where(I['I_TEST1'].notnull())[0]
# Print the index of the OOC points
print('The index of the OOC point is: {}'.format(OOC_idx))

In [ ]:
# Let's find the array of indexes corresponding to OOC points in the I chart for MR

# Find the index of the I_TEST1 column different from NaN
OOC_idx = np.where(I_MR['I_TEST1'].notnull())[0]
# Print the index of the OOC points
print('The index of the OOC point is: {}'.format(OOC_idx))

> We must remove observations with index 18 and 30 from mm^2.

In [ ]:
# Let's substitute the OOC points with NaN

# make a copy of the data
data_cleaned = data.copy()
# replace the OOC point with NaN
data_cleaned.iloc[18] = np.nan
data_cleaned.iloc[30] = np.nan

> Let's recompute control limits.

In [ ]:
# Design the I control chart with the new K value
I = qda.ControlCharts.I(data_cleaned, 'mm^2', K = K_alpha)

In [ ]:
# Extract the 'MR' column   
MR = pd.DataFrame(I['MR'])
# Transform the data
MR['MR_transformed'] = MR['MR'].transform(lambda x: ((x**0.4)))

In [ ]:
I_MR = qda.ControlCharts.I(MR, 'MR_transformed', K = K_alpha)

> The process in in-control